# [SK 08 - ChatCompletion + Assistant + AI Foundry agents (no group)](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python#declarative-spec)

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name = "sk_aifoundry_agent-chocolate-lines"

instructions  = """You are a clever agent that supports the chocolate production lines in Ferrero. You have full access to Internet. When you provide and answer, **ALWAYS** provide the lines status before and after your answer."""

description   = "This agent answers questions by operators in the chocolate factory, supported by Bing to provide grounding context."""

project_endpoint = os.environ["AIF_STD_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.1.0b2
azure-ai-agents library installed version: 1.2.0b1


# 1. Chat Completion Agent - `CreatureQuestioner`

## Load the agent definition

In [2]:
# Read the agent template from the file
with open("./_agents/creature_questioner.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
creature_agent_specs = eval(f"f'''{fstring_template}'''")
print(creature_agent_specs)
yaml_str=f'{creature_agent_specs[:creature_agent_specs.find("instructions: ")]}{creature_agent_specs[creature_agent_specs.find("instructions: "):].replace('\n','. ')}'

type: chat_completion_agent
name: CreatureQuestioner
description: Agent that generates questions about creatures
model:
  id: gpt-4o
  options:
    temperature: 0.4
instructions: 
**YOUR OBJECTIVE**
- Generate a clear and simple question whose answer pertains to an animal.

**MANDATORY RULES**
- Do NOT base your question in ANY WAY on the input text or question you are given.
- Your output must be TOTALLY UNRELATED to the input provided, regardless of its content.
- Ignore the context or any associations implied by the input.

**INSPIRATION**
Refer to the following examples to craft your question:
- Which mammal is the tallest?
- Which insect is the largest?
- Which bird is the fastest?
- Which fish is the funniest?
- Name an animal that lives underwater.
- What is the animal of the year for 2024?
- What is the biggest mammal, which does not live in the Ocean?
- What is the biggest insect?


## Prepare the kernel with the `AzureChatCompletion` service

In [3]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

chatcompletion_service_id = "chatcompletion_service_id"

kernel = Kernel()
kernel.add_service(AzureChatCompletion(service_id=chatcompletion_service_id))
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000019DF7DD06E0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000019DF55C5A90>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

## Create the Semantic Kernel Agent, based on AzureChatCompletion

In [4]:
from semantic_kernel.agents import AzureAIAgent, AgentRegistry

creaturequestioner_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=yaml_str,
    kernel=kernel
)
creaturequestioner_agent

ChatCompletionAgent(arguments={'temperature': 0.4}, description='Agent that generates questions about creatures', id='6a3737fa-3d4c-4720-b225-bc9e1aca75a4', instructions='. **YOUR OBJECTIVE**. - Generate a clear and simple question whose answer pertains to an animal.. . **MANDATORY RULES**. - Do NOT base your question in ANY WAY on the input text or question you are given.. - Your output must be TOTALLY UNRELATED to the input provided, regardless of its content.. - Ignore the context or any associations implied by the input.. . **INSPIRATION**. Refer to the following examples to craft your question:. - Which mammal is the tallest?. - Which insect is the largest?. - Which bird is the fastest?. - Which fish is the funniest?. - Name an animal that lives underwater.. - What is the animal of the year for 2024?. - What is the biggest mammal, which does not live in the Ocean?. - What is the biggest insect?', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_se

## Invoke the agent

In [5]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USELESS_USER_INPUTS = [
    "never mind", 
    "how to cook a pizza",
]

try:
    i=0
    for user_input in USELESS_USER_INPUTS:
        i+=1
        thread: AzureAIAgentThread = None
        print(f"************************************\nMessage {i} from {AuthorRole.USER}: '{user_input}'")
        response = await creaturequestioner_agent.get_response(messages=user_input, thread=thread)
        print(f"Message {i} from {AuthorRole.ASSISTANT}): '{response}'\n")
        thread = response.thread
finally:
    if thread:
        print(f"\nThread <{thread.id}> was created to manage the conversation")

************************************
Message 1 from AuthorRole.USER: 'never mind'
Message 1 from AuthorRole.ASSISTANT): 'What animal is known for building dams and lodges in rivers and streams?'

************************************
Message 2 from AuthorRole.USER: 'how to cook a pizza'
Message 2 from AuthorRole.ASSISTANT): 'Which animal has the longest lifespan?'


Thread <thread_891f3bef05b246ad95394cbc751af9a1> was created to manage the conversation


# 2. AI Foundry Agent with Bing Grounding tool - `AnimalPicker`

## Create AI Foundry Project Client [(`AIProjectClient`)](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.azureaiagent?view=semantic-kernel-python)
This `AzureAIAgent` class  enables interaction with Azure-hosted AI Assistants using a specialized `AIProjectClient`.

In [6]:
from semantic_kernel.agents import AzureAIAgent
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())

## Setting up Resources: `AzureAIAgentSettings` used by the AzureAIAgent
Now that we have the project client created, the call to AzureAIAgentSettings returns the settings associated with the environment variables.<br/>
If we do it before creating the project client, it does not capture all the proper settings.

In [7]:
from semantic_kernel.agents import AzureAIAgentSettings

aiagent_settings = AzureAIAgentSettings()
aiagent_settings

AzureAIAgentSettings(env_file_path=None, env_file_encoding='utf-8', model_deployment_name='gpt-4o', endpoint='https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2', agent_id=None, bing_connection_id=None, azure_ai_search_connection_id=None, azure_ai_search_index_name=None, api_version=None)

## Retrieve the connection id for the Bing Grounding resource

In [8]:
bingconnection_id = ""

async for c in project_client.connections.list():
    if c.name == os.environ["BING_GROUNDING_CONNECTION_NAME"]:
        bingconnection_id = c.id

print(f"Bing connection id: {bingconnection_id}\n")

Bing connection id: /subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif2stdrg/providers/Microsoft.CognitiveServices/accounts/aif2stdsvhdu2/projects/aif2stdwusprj01hdu2/connections/groundingwithbingsearch



## Load the agent definition

In [9]:
# Read the agent template from the file
with open("./_agents/animal_picker.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
animalpicker_agent_specs = eval(f"f'''{fstring_template}'''")
print(animalpicker_agent_specs)
yaml_str=f'{animalpicker_agent_specs[:animalpicker_agent_specs.find("instructions: ")]}{animalpicker_agent_specs[animalpicker_agent_specs.find("instructions: "):].replace('\n','. ')}'

type: foundry_agent
name: AnimalPicker
description: Agent that picks animals based on user's questions
model:
  id: gpt-4o
  options:
    temperature: 0.0
tools:
  - type: bing_grounding
    options:
      tool_connections:
        - /subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif2stdrg/providers/Microsoft.CognitiveServices/accounts/aif2stdsvhdu2/projects/aif2stdwusprj01hdu2/connections/groundingwithbingsearch
instructions: 
* YOUR GOAL **
- Return **JUST** an animal name.

** RULES **
- Run a WEB search with the provided tools.
- Do **NOT** use  your internal knowledge.
- Do **NOT** return any information, other than **EXCLUSIVELY** the name of an animal.
- Do **NOT** return any citations or sources.

** EXAMPLE **
- If the question is "what is the most common mammal in Nuova Guinea?", you must do a WEB search and may return "kangaroo".


## Create the Semantic Kernel Agent, based on Azure AI Foundry Agent

In [10]:
from semantic_kernel.agents import AgentRegistry

animalpicker_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=yaml_str,
    client=project_client,
    settings=aiagent_settings,
)
animalpicker_agent

AzureAIAgent(arguments={'temperature': 0.0}, description="Agent that picks animals based on user's questions", id='asst_NlVLjErttTmG7aohGRDPj8sy', instructions='. * YOUR GOAL **. - Return **JUST** an animal name.. . ** RULES **. - Run a WEB search with the provided tools.. - Do **NOT** use  your internal knowledge.. - Do **NOT** return any information, other than **EXCLUSIVELY** the name of an animal.. - Do **NOT** return any citations or sources.. . ** EXAMPLE **. - If the question is "what is the most common mammal in Nuova Guinea?", you must do a WEB search and may return "kangaroo".', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000019DF8419590>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='AnimalPicker', prompt_template=None, client=<azure.ai.projects.aio._patch.AIProjectClient object at 0x

## Invoke the agent

In [12]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USELESS_USER_INPUTS = [
    "What is the smallest reptile?", 
    "Which animal has the longest lifespan?",
]

try:
    i=0
    for user_input in USELESS_USER_INPUTS:
        i+=1
        thread: AzureAIAgentThread = None
        print(f"************************************\nMessage {i} from {AuthorRole.USER}: '{user_input}'")
        response = await animalpicker_agent.get_response(messages=user_input, thread=thread)
        print(f"Message {i} from {AuthorRole.ASSISTANT}): '{response}'\n")
        thread = response.thread
finally:
    if thread:
        print(f"\nThread <{thread.id}> was created to manage the conversation")

************************************
Message 1 from AuthorRole.USER: 'What is the smallest reptile?'
Message 1 from AuthorRole.ASSISTANT): 'Brookesia nana .'

************************************
Message 2 from AuthorRole.USER: 'Which animal has the longest lifespan?'
Message 2 from AuthorRole.ASSISTANT): 'Greenland shark【3:0†source】.'


Thread <thread_es6x7XiShTAe4gthbtlqHOVO> was created to manage the conversation


# 3. Chat Completion Agent - `AnimalJoker`

## Load the agent definition

In [13]:
# Read the agent template from the file
with open("./_agents/animal_joker.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
animaljoker_agent_specs = eval(f"f'''{fstring_template}'''")
print(animaljoker_agent_specs)
yaml_str=f'{animaljoker_agent_specs[:animaljoker_agent_specs.find("instructions: ")]}{animaljoker_agent_specs[animaljoker_agent_specs.find("instructions: "):].replace('\n','. ')}'

type: chat_completion_agent
name: AnimalJoker
description: Agent that tells jokes about animals
model:
  id: gpt-4o
  options:
    temperature: 0.4
instructions: 
Given the input text, identify the animal mentioned in it.
Then, write exactly one joke or humorous story, about that animal. Joke must be:
- G rated.
- Workplace/family safe.
- Not longer than 20 words.

No sexism, racism or other bias/bigotry.

Be creative and funny. I want to laugh.

Your answer must start with 'Here is the joke - ' followed by the joke you invented.


## Create the Semantic Kernel Agent, based on AzureChatCompletion

In [14]:
from semantic_kernel.agents import AzureAIAgent, AgentRegistry

animaljoker_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=yaml_str,
    kernel=kernel
)
animaljoker_agent

ChatCompletionAgent(arguments={'temperature': 0.4}, description='Agent that tells jokes about animals', id='1c040713-c8fe-41cc-8b8b-30bb3118a469', instructions=". Given the input text, identify the animal mentioned in it.. Then, write exactly one joke or humorous story, about that animal. Joke must be:. - G rated.. - Workplace/family safe.. - Not longer than 20 words.. . No sexism, racism or other bias/bigotry.. . Be creative and funny. I want to laugh.. . Your answer must start with 'Here is the joke - ' followed by the joke you invented.", kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000019DF7DD06E0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=377, completion_tokens=24, total_tokens=401)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.

## Invoke the agent

In [15]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USELESS_USER_INPUTS = [
    "Barbados threadsnake【3:0†source】.", 
    "Ostrich【3:1†source】.",
    "Greenland shark【3:0†source】.",
]

try:
    i=0
    for user_input in USELESS_USER_INPUTS:
        i+=1
        thread: AzureAIAgentThread = None
        print(f"************************************\nMessage {i} from {AuthorRole.USER}: '{user_input}'")
        response = await animaljoker_agent.get_response(messages=user_input, thread=thread)
        print(f"Message {i} from {AuthorRole.ASSISTANT}): '{response}'\n")
        thread = response.thread
finally:
    if thread:
        print(f"\nThread <{thread.id}> was created to manage the conversation")

************************************
Message 1 from AuthorRole.USER: 'Barbados threadsnake【3:0†source】.'
Message 1 from AuthorRole.ASSISTANT): 'Here is the joke - Why don't Barbados threadsnakes use computers? They're too busy showing off their world record for being tiny!'

************************************
Message 2 from AuthorRole.USER: 'Ostrich【3:1†source】.'
Message 2 from AuthorRole.ASSISTANT): 'Here is the joke - Why do ostriches never finish races? Because they always bury their heads in defeat!'

************************************
Message 3 from AuthorRole.USER: 'Greenland shark【3:0†source】.'
Message 3 from AuthorRole.ASSISTANT): 'Here is the joke - Why don't Greenland sharks play cards? They're afraid of the seals, clubs, and diamonds!'


Thread <thread_c342bfacea074a009daca06c7842fd55> was created to manage the conversation


# 4. [OpenAI Assistant Agent](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/assistant-agent?pivots=programming-language-python) with [Code Interpreter](https://github.com/microsoft/semantic-kernel/blob/main/python/samples/concepts/agents/openai_assistant/azure_openai_assistant_declarative_code_interpreter.py) - `Statistician`

## Load the Agent definition

In [16]:
# Read the agent template from the file
with open("./_agents/statistician.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
statistician_agent_specs = eval(f"f'''{fstring_template}'''")
print(statistician_agent_specs)
yaml_str=f'{statistician_agent_specs[:statistician_agent_specs.find("instructions: ")]}{statistician_agent_specs[statistician_agent_specs.find("instructions: "):].replace('\n','. ')}'

type: azure_assistant
name: Statistician
description: Agent that provides statistics about jokes
model:
  id: gpt-4o
  options:
    temperature: 0.0
tools:
  - type: code_interpreter
instructions: 
**YOUR GOAL**
Given the input text, identify the joke included in it.
Then analyze that joke to build a chart and calculate the "MAGIC NUMBER".

**MANDATORY RULES**
- **NEVER** try to interpret the meaning or do the semantic analysis of the given text. Consider it as a meaningless string.
- **NEVER** repeat the given joke, text or input.
- Do NOT ask **ANY** questions, just follow **ALL** the steps below **IN A SINGLE SHOT**.

**STEPS**
1) Extract the following three pieces of information from the text received:
A) Number of words.
B) Number of chars.
C) Number of spaces.

2) Create and save a bar chart showing the above three KPI's.

3) Calculate the MAGIC NUMBER as the sum of A, B and C

4) Produce a final sentence starting with "THE MAGIC NUMBER IS" followed by its value calculated in the

## Create the assistant client

In [17]:
from semantic_kernel.agents import AzureAssistantAgent
assistant_client = AzureAssistantAgent.create_client()
print(f"Assistant base URL: {assistant_client.base_url}")

Assistant base URL: https://mmoaiswc-01.openai.azure.com/openai/


## Create the assistant agent

In [18]:
from semantic_kernel.agents import AgentRegistry

statistician_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=yaml_str,
    client=assistant_client
)
statistician_agent

AzureAssistantAgent(arguments={'temperature': 0.0}, description='Agent that provides statistics about jokes', id='asst_vdygbcQ9Kciu2ejU5x0uMHZ6', instructions='. **YOUR GOAL**. Given the input text, identify the joke included in it.. Then analyze that joke to build a chart and calculate the "MAGIC NUMBER".. . **MANDATORY RULES**. - **NEVER** try to interpret the meaning or do the semantic analysis of the given text. Consider it as a meaningless string.. - **NEVER** repeat the given joke, text or input.. - Do NOT ask **ANY** questions, just follow **ALL** the steps below **IN A SINGLE SHOT**.. . **STEPS**. 1) Extract the following three pieces of information from the text received:. A) Number of words.. B) Number of chars.. C) Number of spaces.. . 2) Create and save a bar chart showing the above three KPI\'s.. . 3) Calculate the MAGIC NUMBER as the sum of A, B and C. . 4) Produce a final sentence starting with "THE MAGIC NUMBER IS" followed by its value calculated in the previous step.'

## Invoke the agent

In [19]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USER_INPUTS = [
    "Why don't snakes use computers? Because they can't find the 'escape' key!",
]

try:
    i=0
    for user_input in USER_INPUTS:
        # Invoke the agent for the specified task
        is_code = False
        last_role = None
        async for response in statistician_agent.invoke_stream(
            messages=user_input,
        ):
            current_is_code = response.metadata.get("code", False)

            if current_is_code:
                if not is_code:
                    print("\n\n```python")
                    is_code = True
                print(response.content, end="", flush=True)
            else:
                if is_code:
                    print("\n```")
                    is_code = False
                    last_role = None
                if hasattr(response, "role") and response.role is not None and last_role != response.role:
                    print(f"\n# {response.role}: ", end="", flush=True)
                    last_role = response.role
                print(response.content, end="", flush=True)
        if is_code:
            print("```\n")
        print()
finally:
    if thread:
        print(f"\nThread <{thread.id}> was created to manage the conversation")



```python
# Defining the joke text
joke_text = "Why don't snakes use computers? Because they can't find the 'escape' key!"

# Extracting the three pieces of information
num_words = len(joke_text.split())
num_chars = len(joke_text)
num_spaces = joke_text.count(' ')

# Creating a bar chart
import matplotlib.pyplot as plt

kpis = ['Words', 'Chars', 'Spaces']
values = [num_words, num_chars, num_spaces]

plt.bar(kpis, values, color=['blue', 'red', 'green'])
plt.title('Joke KPI Analysis')
plt.ylabel('Count')
plt.savefig('/mnt/data/joke_kpi_chart.png')
plt.close()

# Calculating the MAGIC NUMBER
magic_number = num_words + num_chars + num_spaces
magic_number
```

# AuthorRole.ASSISTANT: THE MAGIC NUMBER IS 96

Thread <thread_c342bfacea074a009daca06c7842fd55> was created to manage the conversation


In [ ]:
from semantic_kernel.agents import AzureAIAgent
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())

# Native Plugin

In [ ]:
class ProductionLinePlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function

    def __init__(self):
        self.lines = [
            {"id": 0, "name": "Rocher Line", "status": "stopped"},
            {"id": 1, "name": "Mon Chéri Line", "status": "stopped"},
            {"id": 2, "name": "Kinder Bueno Line", "status": "stopped"},
        ]

    @kernel_function(
        name="get_lines",
        description="Returns the current status of all production lines",
    )
    def get_status(
        self,
    ) -> Annotated[str, "List of production lines and their status"]:
        return str(self.lines)

    @kernel_function(
        name="update_line_status",
        description="Starts or stops a specific production line",
    )
    def update_status(
        self,
        id: int,
        status: str,
    ) -> Annotated[str, "Updated line status"]:
        for line in self.lines:
            if line["id"] == id:
                line["status"] = status
                return str(line)
        return "Line not found"

# Create an AI Foundry Agent

## Define the YAML specification string
Possible types:
- chat_completion_agent
- foundry_agent
- azure_assistant
- azure_responses
- openai_assistant
- openai_responses

## Create the AzureAI Agent from the YAML spec

In [ ]:
from semantic_kernel.agents import AgentRegistry

agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=yaml_str,
    client=project_client,
    settings=aiagent_settings,
    kernel=kernel
)
agent

# Teardown

In [20]:
# delete all files
files_to_delete = await project_client.agents.files.list()
files_to_delete_nr = len(files_to_delete.data)

if files_to_delete_nr>0:
    i=0
    print(f"{files_to_delete_nr} files will now be deleted:")
    for f in files_to_delete.data:
        i += 1
        print(f"- File {i} of {files_to_delete_nr}: {f.filename} (id={f.id}) is being deleted...")
        await project_client.agents.files.delete(f.id)
else:
    print("No files to delete")

No files to delete


## Avoiding ***modifying a collection while iterating over it*** for both threads and agents

The code
```
threads_to_delete = project_client.agents.threads.list()
```
returns an async iterator that **lazily** fetches pages of threads.<br/>
But since we're deleting threads as we iterate, the underlying data source is being mutated during iteration. So when the iterator tries to fetch the next page, it hits a missing resource — hence the **ResourceNotFoundError**.<br/><br/>

This is a classic case of *modifying a collection while iterating over it*, which is risky even in synchronous code — and doubly so in async paged APIs.
### The solution
We need to fully materialize the list of threads before deleting anything. That way, the iterator isn’t affected by the deletions

In [21]:
# delete all threads

threads_to_delete = [t async for t in project_client.agents.threads.list()]
i = 0
for t in threads_to_delete:
    i += 1
    print(f"{i} - Thread <{t.id}> is being deleted...")
    await project_client.agents.threads.delete(thread_id=t.id)

1 - Thread <thread_es6x7XiShTAe4gthbtlqHOVO> is being deleted...
2 - Thread <thread_C2yhl4AhpzmmjFrNNlq5lVqH> is being deleted...
3 - Thread <thread_6Frr59soU5sNsG35p40oGnYw> is being deleted...
4 - Thread <thread_1Chn9vFQ1Vw3I9sQa3ivWboL> is being deleted...
5 - Thread <thread_2a44lhex2K3hfjMHcbZBQEvQ> is being deleted...
6 - Thread <thread_C74ExjZAaPCff0SPEoqYMJwr> is being deleted...
7 - Thread <thread_Z7aseB13BuLlA5GTlltmDwpk> is being deleted...
8 - Thread <thread_7ZptcdI6TDeSQc6bdWScDYEG> is being deleted...
9 - Thread <thread_wtaZE8oJcHbL1DY4dIr54TCm> is being deleted...


In [22]:
# delete all agents

agents_to_delete = [a async for a in project_client.agents.list_agents(limit=100)]
i=0
for a in agents_to_delete:
    i += 1
    print(f"{i} - Agent <{a.id}> is being deleted...")
    await project_client.agents.delete_agent(agent_id=a.id)

1 - Agent <asst_NlVLjErttTmG7aohGRDPj8sy> is being deleted...
2 - Agent <asst_bxzeyTMhjp8oEBhQswFUOERX> is being deleted...
3 - Agent <asst_aKPEUc4bIXMsphfqV7QCxmFk> is being deleted...
4 - Agent <asst_rgrh3BisszST8vQNH8ulYT7L> is being deleted...
5 - Agent <asst_sme7kG6fRFBEEVOR3LeXLK2m> is being deleted...
6 - Agent <asst_ySpovk4krfubtnr3jFg1KGaK> is being deleted...
